In [1]:
from torchvision.models import MobileNet_V3_Large_Weights

weights = MobileNet_V3_Large_Weights.DEFAULT

print(weights.transforms())

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [2]:
from torchvision import transforms

IMAGE_SIZE = 512

normalize = transforms.Normalize(
    mean=weights.transforms().mean,
    std=weights.transforms().std
)

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    normalize
])

In [3]:
from pathlib import Path
from torchvision import datasets

DATA_DIR = Path("../dataset/Original CT Scans")

dataset = datasets.ImageFolder(
    root=str(DATA_DIR),
    transform=transform
)

image, label = dataset[0]

print("Image shape:", image.shape)
print("Label:", dataset.classes[label])
print("Pixel tensor dtype:", image.dtype)

Image shape: torch.Size([3, 512, 512])
Label: NiCT
Pixel tensor dtype: torch.float32


In [4]:
from sklearn.model_selection import train_test_split
import numpy as np
from torch.utils.data import Subset


indices = np.arange(len(dataset))
labels = np.array(dataset.targets)

train_indices, temp_indices = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=42,
    stratify=labels[temp_indices]
)

train_dataset = Subset(dataset, train_indices.tolist())
val_dataset = Subset(dataset, val_indices.tolist())
test_dataset = Subset(dataset, test_indices.tolist())

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Training: 13779
Validation: 2953
Test: 2953


In [5]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 862
Validation batches: 185
Test batches: 185


In [6]:
from torchvision import transforms

IMAGE_SIZE = 512

data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),

    "eval": transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}

In [8]:
from torchvision import datasets
from torch.utils.data import DataLoader, Subset

base_dataset = datasets.ImageFolder(
    root=str(DATA_DIR)
)

train_dataset = datasets.ImageFolder(
    root=str(DATA_DIR),
    transform=data_transforms["train"]
)

eval_dataset = datasets.ImageFolder(
    root=str(DATA_DIR),
    transform=data_transforms["eval"]
)

train_dataset = Subset(train_dataset, train_indices.tolist())
val_dataset = Subset(eval_dataset, val_indices.tolist())
test_dataset = Subset(eval_dataset, test_indices.tolist())